# Train CNN–LSTM trên `obfu_payload.csv` bằng Google Colab

Notebook này chạy đúng pipeline của repo (`preprocessing/preprocess_data.py` → `cnn_lstm/CNN_LSTM.py`) trên GPU Colab, chỉ với source `obfu_payload`.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`.

Các bước:
1. Clone repo.
2. Nạp `obfu_payload.csv` (từ Google Drive hoặc upload trực tiếp).
3. Train.
4. Xem kết quả và lưu artifacts về Drive.


## 1. Kiểm tra GPU và clone repo

In [ ]:
!nvidia-smi -L

import os
REPO = "https://github.com/khangdz296/obfuscated-web-attack-detection.git"
BRANCH = "main"
PROJECT_DIR = "/content/obfuscated-web-attack-detection"

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {BRANCH} {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull --ff-only
%cd {PROJECT_DIR}


## 2. Nạp dataset

Dataset không nằm trong git (`*.csv` bị ignore), nên phải đưa file vào `DataSet/obfu_payload.csv`. Chọn **một** trong hai cách bên dưới.

**Cách A – Google Drive (khuyên dùng, file 25 MB upload lại mỗi phiên rất chậm):** upload `obfu_payload.csv` lên Drive một lần, rồi chỉnh `DRIVE_CSV` cho đúng đường dẫn.

In [ ]:
# Cách A: copy từ Google Drive
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CSV = "/content/drive/MyDrive/obfu_payload.csv"   # <-- chỉnh đường dẫn nếu khác

os.makedirs("DataSet", exist_ok=True)
!cp "{DRIVE_CSV}" DataSet/obfu_payload.csv
!ls -lh DataSet/


In [ ]:
# Cách B: upload trực tiếp từ máy (bỏ qua nếu đã dùng cách A)
# from google.colab import files
# os.makedirs("DataSet", exist_ok=True)
# uploaded = files.upload()          # chọn obfu_payload.csv
# for name in uploaded:
#     os.replace(name, "DataSet/obfu_payload.csv")
# !ls -lh DataSet/


## 3. Cài dependencies

Colab đã có sẵn TensorFlow + GPU, chỉ cần bổ sung vài gói. **Không** cài lại `tensorflow==2.17.1` từ `webapp/requirements.txt` trừ khi bắt buộc – cài lại TF trên Colab mất nhiều phút và có thể làm lệch CUDA.

In [ ]:
!pip -q install scikit-learn pandas numpy
import tensorflow as tf
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))


## 4. Kiểm tra nhanh preprocessing

Bước này chỉ vài giây, để chắc loader đọc đúng schema `payload,label,attack_type,technique,family` và split theo `family` không bị rò rỉ.

In [ ]:
!python preprocessing/preprocess_data.py     --datasets obfu_payload     --split-protocol family_group     --output-dir /content/processed_obfu_payload

import pandas as pd
tr = pd.read_csv("/content/processed_obfu_payload/obfu_payload/train.csv")
te = pd.read_csv("/content/processed_obfu_payload/obfu_payload/test.csv")
print("family chung giữa train và test:", len(set(tr.split_group) & set(te.split_group)))
print(tr.payload.iloc[0][:200])


## 5. Train

Tham số chính:
- `--split-protocol family_group`: các biến thể obfuscate của cùng một seed payload không bao giờ vừa nằm ở train vừa nằm ở test → đánh giá thật hơn `random_stratified_row`.
- `--max-len 1024`: sau khi bọc vào HTTP envelope, p95 độ dài ≈ 850 ký tự, p99 ≈ 1090; mặc định 768 sẽ cắt mất ~5% mẫu.
- `--epochs 30`: có EarlyStopping theo `val_loss`, thường dừng sớm hơn.

Muốn chạy thử nhanh trước, thêm `--obfu-sample-size 10000 --epochs 3`.

In [ ]:
ARTIFACT_DIR = "/content/artifacts_obfu_payload"

!python cnn_lstm/CNN_LSTM.py     --datasets obfu_payload     --train-sources obfu_payload     --split-protocol family_group     --max-len 1024     --batch-size 256     --epochs 30     --output-dir {ARTIFACT_DIR}


## 6. Xem kết quả

In [ ]:
import json
from pathlib import Path

meta = json.loads((Path(ARTIFACT_DIR) / "by_dataset/obfu_payload/metadata_and_results.json").read_text(encoding="utf-8"))
for name, ev in meta["evaluation"].items():
    print(f"== {name} ==")
    for k in ("accuracy", "auc_roc", "pr_auc", "decision_threshold"):
        print(f"  {k:18s} {ev[k]:.4f}")
    print("  confusion_matrix  ", ev["confusion_matrix"])
    attack = ev["classification_report"].get("Attack (1)", {})
    print(f"  attack P/R/F1      {attack.get('precision', 0):.4f} / {attack.get('recall', 0):.4f} / {attack.get('f1-score', 0):.4f}")

display(pd.read_csv(Path(ARTIFACT_DIR) / "cross_eval_results.csv"))
display(pd.read_csv(Path(ARTIFACT_DIR) / "by_dataset/obfu_payload/training_history.csv").tail())


## 7. Lưu artifacts về Drive

Colab xoá `/content` khi hết phiên. Model + tokenizer nằm ở `by_dataset/obfu_payload/`, cần copy về Drive để dùng cho webapp (`cnn_lstm/artifacts_cnn_lstm_by_dataset/by_dataset/<source>/`).

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/artifacts_obfu_payload"
!mkdir -p "{DRIVE_OUT}"
!cp -r {ARTIFACT_DIR}/. "{DRIVE_OUT}/"
!ls -lh "{DRIVE_OUT}/by_dataset/obfu_payload"

# Hoặc tải thẳng về máy:
# !zip -qr /content/artifacts_obfu_payload.zip {ARTIFACT_DIR}
# from google.colab import files; files.download("/content/artifacts_obfu_payload.zip")
